In [1]:
import sys
print(sys.executable)

/home/zvj3hf/.conda/envs/py311/bin/python


In [2]:
conda install numba cudatoolkit -y

Jupyter detected...
Retrieving notices: done
Channels:
 - conda-forge
Platform: linux-64
Solving environment: done

## Package Plan ##

  environment location: /home/zvj3hf/.conda/envs/py311

  added / updated specs:
    - cudatoolkit
    - numba


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2026.4.22  |       hbd8a1cb_0         128 KB  conda-forge
    certifi-2026.4.22          |     pyhd8ed1ab_0         132 KB  conda-forge
    cudatoolkit-11.8.0         |      h4ba93d1_13       682.5 MB  conda-forge
    ------------------------------------------------------------
                                           Total:       682.7 MB

The following NEW packages will be INSTALLED:

  cudatoolkit        conda-forge/linux-64::cudatoolkit-11.8.0-h4ba93d1_13 

The following packages will be UPDATED:

  ca-certificates                      2026.2.25-hbd8a1cb_0 --> 2026.4.22-hbd8

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import airy, gamma
from matplotlib.colors import LogNorm
import mpmath as mp

from numba import cuda
import math
import time

# ─────────────────────────────────────────────────────────────────────────────
# 1) Timing start
start = time.time()

# 2) Problem size & splice radius
N, cut = 400, 8.0

# 3) CUDA launch configuration
threadsperblock = (16, 16)
blockspergrid   = (
    math.ceil(N / threadsperblock[0]),
    math.ceil(N / threadsperblock[1]),
)
print(f"Launching {blockspergrid} blocks of {threadsperblock} threads each")

# 4) Build host grid and send real/imag parts to GPU
x = np.linspace(-10, 10, N)
y = np.linspace(-10, 10, N)
Xh, Yh = np.meshgrid(x, y)
Zr_dev = cuda.to_device(Xh)       # real part on device
Zi_dev = cuda.to_device(Yh)       # imag part on device

# ─────────────────────────────────────────────────────────────────────────────
# 5) Precompute Taylor‐series coefficients on host, then send to GPU
terms = 150
coeffs_host = np.zeros(terms, dtype=np.complex128)
coeffs_host[0] = 1.0 / (3**(2/3) * gamma(2/3))
coeffs_host[1] = -1.0 / (3**(1/3) * gamma(1/3))
for k in range(2, terms):
    coeffs_host[k] = coeffs_host[k-3] / (k*(k-1))
coeffs_dev = cuda.to_device(coeffs_host)

# 6) CUDA kernel: compute |Ai(z)| via Horner’s method on device
@cuda.jit
def compute_mag_exact_kernel(Zr, Zi, coeffs, mag):
    i, j = cuda.grid(2)
    if i < mag.shape[0] and j < mag.shape[1]:
        # reconstruct complex z
        zr = Zr[i, j]
        zi = Zi[i, j]
        # Horner initialization: last coefficient
        sr = coeffs[terms-1].real
        si = coeffs[terms-1].imag
        # Horner loop: s = s*z + c
        for kk in range(terms-2, -1, -1):
            cr = coeffs[kk].real
            ci = coeffs[kk].imag
            tr = sr*zr - si*zi + cr
            ti = sr*zi + si*zr + ci
            sr, si = tr, ti
        # magnitude = sqrt(sr^2 + si^2)
        mag[i, j] = math.sqrt(sr*sr + si*si)

# 7) Allocate device array for result and launch kernel
mag_exact_dev = cuda.device_array((N, N), dtype=np.float64)
compute_mag_exact_kernel[blockspergrid, threadsperblock](
    Zr_dev, Zi_dev, coeffs_dev, mag_exact_dev
)
cuda.synchronize()

# 8) Copy back the fully computed |Ai_exact| once
mag_exact = mag_exact_dev.copy_to_host()

# ─────────────────────────────────────────────────────────────────────────────
# 9) Taylor Approx (still on host)
def airy_taylor(zv):
    out = np.zeros_like(zv, dtype=complex)
    it = np.nditer(zv, flags=['multi_index'])
    while not it.finished:
        z = it[0]
        s = coeffs_host[-1]
        for c in coeffs_host[-2::-1]:
            s = s*z + c
        out[it.multi_index] = s
        it.iternext()
    return out

# 10) Asymptotic Approx setup
N1 = 50
u1_host = np.ones(N1, dtype=complex)
for k in range(1, N1):
    u1_host[k] = ((6*k-5)*(6*k-1))/(48*k) * u1_host[k-1]

N2 = 40
u2_host = np.ones(N2, dtype=complex)
for k in range(1, N2):
    u2_host[k] = ((6*k-5)*(6*k-1))/(48*k) * u2_host[k-1]

def airy_asymp_pos(zv):
    z = np.asarray(zv, dtype=complex)
    ζ = (2/3)*z**1.5
    S = sum(((-1)**k)*u1_host[k]/ζ**k for k in range(N1))
    return np.exp(-ζ)/(2*np.sqrt(np.pi)*z**0.25) * S

def airy_asymp_neg(zv):
    w = -zv
    ζ = (2/3)*w**1.5
    phi = ζ - np.pi/4
    even = sum(((-1)**k)*u2_host[2*k]   / ζ**(2*k)   for k in range(N2//2))
    odd  = sum(((-1)**k)*u2_host[2*k+1] / ζ**(2*k+1) for k in range((N2-1)//2))
    return (1/np.sqrt(np.pi)) * w**(-1/4) * (np.cos(phi)*even - np.sin(phi)*odd)

# 11) Build Z on host for these host‐based funcs
Z_host = Xh + 1j*Yh
Ai_mid   = airy_taylor(Z_host)
Ai_tail  = np.where(Z_host.real>=0,
                    airy_asymp_pos(Z_host),
                    airy_asymp_neg(Z_host))
Ai_splice= np.where(np.abs(Z_host)<=cut, Ai_mid, Ai_tail)
mag_splice = np.abs(Ai_splice)

# 12) Find zeros on real axis
x_host = x
exact_line = airy(x_host)[0]
mid_line   = airy_taylor(x_host)
tail_line  = np.where(x_host>=0,
                     airy_asymp_pos(x_host),
                     airy_asymp_neg(x_host))
splice_line = np.where(np.abs(x_host)<=cut, mid_line, tail_line)

def find_roots(f, xs):
    idx = np.where(np.sign(f[:-1])*np.sign(f[1:])<0)[0]
    roots = []
    for i in idx:
        x0, x1 = xs[i], xs[i+1]
        y0, y1 = f[i], f[i+1]
        roots.append(x0 - y0*(x1-x0)/(y1-y0))
    return roots

roots_exact  = find_roots(np.real(exact_line), x_host)
roots_splice = find_roots(np.real(splice_line), x_host)

df = pd.DataFrame({
    'Exact zeros':   roots_exact,
    'Splice zeros':  roots_splice
})
print("\nEstimated real zeros of Ai(z):")
print(df.to_string(index=False))

# ─────────────────────────────────────────────────────────────────────────────
# 13) Plotting
norm = LogNorm(vmin=1e-5, vmax=1e1)
fig, axs = plt.subplots(1, 2, figsize=(12, 6), constrained_layout=True)
pcm0 = axs[0].pcolormesh(Xh, Yh, mag_exact, norm=norm, shading='auto', cmap='viridis')
axs[0].set_title('Exact |Ai(z)|'); axs[0].set_aspect('equal'); fig.colorbar(pcm0, ax=axs[0])
pcm2 = axs[1].pcolormesh(Xh, Yh, mag_splice, norm=norm, shading='auto', cmap='viridis')
axs[1].set_title('Spliced |Ai(z)|'); axs[1].set_aspect('equal'); fig.colorbar(pcm2, ax=axs[1])
for ax in axs:
    ax.set_xlim(-10, 10); ax.set_ylim(-10, 10)
    ax.set_xlabel('Re(z)'); ax.set_ylabel('Im(z)')
plt.show()

# 14) Timing end
end = time.time()
print(f"Total GPU‐accelerated time: {end-start:.3f} seconds")


Launching (25, 25) blocks of (16, 16) threads each


CudaSupportError: Error at driver init: 

CUDA driver library cannot be found.
If you are sure that a CUDA driver is installed,
try setting environment variable NUMBA_CUDA_DRIVER
with the file path of the CUDA driver shared library.
: